### Import Dependecies

In [1]:
from pydantic import BaseModel, Field

from langgraph.graph import StateGraph, START, END
from langgraph.prebuilt import ToolNode

from langchain_core.messages import AIMessage, ToolMessage, SystemMessage
from langchain_core.messages import convert_to_openai_messages, convert_to_messages

from jinja2 import Template
from typing import Literal, Dict, Any, Annotated, List
from IPython.display import Image, display
from operator import add
from openai import OpenAI
import openai

import random
import ast
import inspect
import instructor
import json


from utils.utils import get_tool_descriptions, format_ai_message
from utils.tools import get_formatted_items_context, get_formatted_reviews_context, add_to_shopping_cart, get_shopping_cart, remove_from_cart, check_warehouse_availability, reserve_warehouse_items

from langsmith import traceable

from langchain_openai import ChatOpenAI
from langchain_core.tools import tool

from langgraph.checkpoint.postgres import PostgresSaver

from qdrant_client import QdrantClient
from qdrant_client import models
from qdrant_client.models import Filter, FieldCondition, MatchValue, VectorParams, Distance, SparseVectorParams, Modifier, PayloadSchemaType, PointStruct, Document, Prefetch, FusionQuery, MatchAny

## Coordinator Agent

In [2]:
class Delegation(BaseModel):
    agent: str
    task: str

class CoordinatorAgentResponse(BaseModel):
    next_agent: str
    plan: List[Delegation]
    final_answer: bool = False
    answer: str

class RAGUsedContext(BaseModel):
    id: str = Field(description="ID of the item used to answer the question")
    description: str = Field(description="Short description of the item used to answer the question")

In [3]:
class ToolCall(BaseModel):
    name: str
    arguments: dict
    
class AgentPreporties(BaseModel):
    iteration: int = 0
    available_tools: List[Dict[str, Any]] = []
    tool_calls: List[ToolCall] = []
    final_answer: bool = False

class CoordinatorAgentPreporties(BaseModel):
    iteration: int = 0
    final_answer: bool = False
    plan: List[Delegation] = []
    next_agent: str = ""

class State(BaseModel):
    messages: Annotated[List[Any], add] = []
    user_intent: str = ""
    product_qa_agent: AgentPreporties = Field(default_factory=AgentPreporties)
    shopping_cart_agent: AgentPreporties = Field(default_factory=AgentPreporties)
    warehouse_manager_agent: AgentPreporties = Field(default_factory=AgentPreporties)
    coordinator_agent: CoordinatorAgentPreporties = Field(default_factory=CoordinatorAgentPreporties)
    answer: str = ""
    references: Annotated[List[RAGUsedContext], add] = []
    user_id: str = ""
    cart_id: str = ""

In [4]:
@traceable(
    name="coordinator_agent",
    run_type="llm",
    metadata={"ls_provider": "openai", "ls_model_name": "gpt-5.4-mini"}
)
def coordinator_agent(state) -> dict:

    prompt_template = """You are a Coordinator Agent as part of a shopping assistant.

Your role is to create plans for solving user queries and delegate the tasks accordingly.
You will be given a conversation history, your task is to create a plan for solving the user's query.
After the plan is created, you should output the next agent to invoke and the task to be performed by that agent.
Once an agent finishes its task, you will be handed the control back, you should then review the conversation history and revise the plan.
If there is a sequence of tasks to be performed by a single agent, you should combine them into a single task.

The possible agents are:

- product_qa_agent: The user is asking a question about a product. This can be a question about available products, their specifications, user reviews etc.
- shopping_cart_agent: The user is asking to add or remove items from the shopping cart or questions about the current shopping cart.
- warehouse_manager_agent: The user is asking to reserve items from the warehouses or about availability of the items in warehouses.

CRITICAL RULES:
- If next_agent is "", final_answer MUST be false
(You cannot delegate the task to an agent and return to the user in the same response)
- If final_answer is true, next_agent MUST be ""
(You must wait for agent results before returning to user)
- If you need to call other agents before answering, set:
next_agent="...", final_answer=false
- After receiving agent results, you can then set:
next_agent="", final_answer=true
- One of the following has to be true:
next_agent is "" and final_answer is true
next_agent is not "" and final_answer is false

Additional instructions:

- Do not route to any agent if the user's query needs clarification. Do it yourself.
- Write the plan to the plan field.
- Write the next agent to invoke to the next_agent field.
- Once you have all the information needed to answer the user's query, you should set the final_answer field to True and output the answer to the user's query.
- The final answer to the user query should be a comprehensive answer that explains the actions that were performed to answer the query.
- Never set final_answer to true if the plan is not complete.
- You should output the next_agent field as well as the plan field.
"""

    template = Template(prompt_template)

    prompt = template.render()

    messages = state.messages

    conversation = []

    for message in messages:
        conversation.append(convert_to_openai_messages(message))

    client = instructor.from_provider(
        "openai/gpt-5.4-mini"
    )

    response, raw_response = client.create_with_completion(
        messages=[
            {"role": "system", "content": prompt},
            *conversation
        ],
        response_model=CoordinatorAgentResponse,
    )

    if response.final_answer:
        ai_message = [AIMessage(
            content=response.answer
        )]
    else:
        ai_message = []
 
    return {
        "messages": ai_message,
        "answer": response.answer,
        "coordinator_agent": {
            "iteration": state.coordinator_agent.iteration + 1,
            "final_answer": response.final_answer,
            "next_agent": response.next_agent,
            "plan": [data.model_dump() for data in response.plan]
        }
    }

In [5]:
initial_state = State(
    messages=[{
        "role": "user",
        "content": "what is the weather in Almaty?"
    }]
)

In [6]:
answer = coordinator_agent(initial_state)

In [7]:
answer

{'messages': [AIMessage(content='I’m sorry, but I can’t help with weather queries in this shopping assistant.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])],
 'answer': 'I’m sorry, but I can’t help with weather queries in this shopping assistant.',
 'coordinator_agent': {'iteration': 1,
  'final_answer': True,
  'next_agent': '',
  'plan': [{'agent': 'product_qa_agent',
    'task': 'The user is asking about the weather in Almaty, but none of the available shopping-related agents can answer weather questions. No delegation is possible.'}]}}

In [8]:
initial_state = State(
    messages=[{
        "role": "user",
        "content": "find me some earphones?"
    }]
)

In [9]:
answer = coordinator_agent(initial_state)

In [10]:
answer

{'messages': [],
 'answer': '',
 'coordinator_agent': {'iteration': 1,
  'final_answer': False,
  'next_agent': 'product_qa_agent',
  'plan': [{'agent': 'product_qa_agent',
    'task': 'Find suitable earphones for the user. If possible, provide a few options across common use cases (e.g., budget, best overall, wireless/wired) and summarize key specs/features to help the user choose.'}]}}

In [11]:
coordinator_eval_dataset = [
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "what is the weather in Almaty?"}
            ]
        },
        "outputs": {
            "next_agent": "",
            "coordinator_final_answer": True
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "can i get some earphones?"}
            ]
        },
        "outputs": {
            "next_agent": "product_qa_agent",
            "coordinator_final_answer": False
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "can you add item with ID B09NLTDHQ6 to my cart?"}
            ]
        },
        "outputs": {
            "next_agent": "shopping_cart_agent",
            "coordinator_final_answer": False
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "can you add those earphones to my cart?"}
            ]
        },
        "outputs": {
            "next_agent": "",
            "coordinator_final_answer": True
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "can you add the best items to my cart? I am looking for a laptop bags."}
            ]
        },
        "outputs": {
            "next_agent": "product_qa_agent",
            "coordinator_final_answer": False
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "can you find some good reviews for items in my cart?"}
            ]
        },
        "outputs": {
            "next_agent": "shopping_cart_agent",
            "coordinator_final_answer": False
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "can you put the items with the most positive user reviews to my cart?"}
            ]
        },
        "outputs": {
            "next_agent": "product_qa_agent",
            "coordinator_final_answer": False
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "what kind of stuff do you sell?"}
            ]
        },
        "outputs": {
            "next_agent": "",
            "coordinator_final_answer": True
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "can you help me with my order?"}
            ]
        },
        "outputs": {
            "next_agent": "",
            "coordinator_final_answer": True
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "can you add two, ideally red tablets to my cart?"}
            ]
        },
        "outputs": {
            "next_agent": "product_qa_agent",
            "coordinator_final_answer": False
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "can you reserve a red leather laptop bag for me?"}
            ]
        },
        "outputs": {
            "next_agent": "product_qa_agent",
            "coordinator_final_answer": False
        }
    },
    {
        "inputs": {
            "messages": [
                {"role": "user", "content": "can you reserve my shopping cart?"}
            ]
        },
        "outputs": {
            "next_agent": "shopping_cart_agent",
            "coordinator_final_answer": False
        }
    },
]

### Upload dataset to Langsmith

In [ ]:
from langsmith import Client
import os

client = Client(api_key=os.environ["LANGSMITH_API_KEY"])

dataset_name = "coordinator_eval_dataset"
dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Coordinator Agent Evaluation Dataset for the first routing step"
)

Failed to get info from https://api.smith.langchain.com: LangSmithConnectionError('Connection error caused failure to GET /info in LangSmith API. Please confirm your internet connection. ConnectTimeout(MaxRetryError("HTTPSConnectionPool(host=\'api.smith.langchain.com\', port=443): Max retries exceeded with url: /info (Caused by ConnectTimeoutError(<HTTPSConnection(host=\'api.smith.langchain.com\', port=443) at 0x760804413d10>, \'Connection to api.smith.langchain.com timed out. (connect timeout=10.0)\'))"))\nContent-Length: None\nAPI Key: lsv2_********************************************8c')
Run compression is not enabled. Please update to the latest version of LangSmith. Falling back to regular multipart ingestion.


In [13]:
for item in coordinator_eval_dataset:
    client.create_example(
        dataset_id=dataset.id,
        inputs={"messages": item["inputs"]["messages"]},
        outputs={
            "next_agent": item["outputs"]["next_agent"],
            "coordinator_final_answer": item["outputs"]["coordinator_final_answer"]
        }
    )

In [14]:
def next_agent_evaluator(run, example):

    next_agent_match = run.outputs["coordinator_agent"]["next_agent"] == example.outputs["next_agent"]
    final_answer_match = run.outputs["coordinator_agent"]["final_answer"] == example.outputs["coordinator_final_answer"]

    return next_agent_match and final_answer_match

In [15]:
results = client.evaluate(
    lambda x: coordinator_agent(State(messages=x["messages"])),
    data="coordinator_eval_dataset",
    evaluators=[
        next_agent_evaluator
    ],
    experiment_prefix="coordinator-eval-dataset"
)

/home/shanks/Projects/ai-taste/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View the evaluation results for experiment: 'coordinator-eval-dataset-ec79efb1' at:
https://smith.langchain.com/o/ba3d2314-c180-4ae1-b336-ae504b26cdf7/datasets/8e18f89e-b354-45f1-96c3-2f2b00009271/compare?selectedSessions=b92a5081-7b7c-4100-aa4d-f32a5f52c2dc




12it [00:33,  2.80s/it]
